# Feature Engineering

This notebook takes the raw Telco Churn dataset and prepares it for modelling.

Steps:
1. Drop customerID — no predictive value
2. Convert binary text columns to 0/1
3. One-hot encode multi-value columns
4. Engineer 4 new features:
   - AvgMonthlySpend — captures spend relative to tenure
   - TotalServices — number of services subscribed to
   - IsNewCustomer — flags customers in their first 12 months
   - HighValueCustomer — flags customers paying above median
   
Output: churn_cleaned.csv — 7043 rows, 35 columns

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the raw data again
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Repeat the two cleaning steps from notebook 1
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Data loaded: 7043 rows, 21 columns


In [2]:
df = df.drop('customerID', axis=1)

print(f"Columns remaining: {df.shape[1]}")
print(df.columns.tolist())

Columns remaining: 20
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [3]:
text_columns = df.select_dtypes(include='object').columns

for col in text_columns:
    print(f"{col}: {df[col].unique()}")
    print()

gender: ['Female' 'Male']

Partner: ['Yes' 'No']

Dependents: ['No' 'Yes']

PhoneService: ['No' 'Yes']

MultipleLines: ['No phone service' 'No' 'Yes']

InternetService: ['DSL' 'Fiber optic' 'No']

OnlineSecurity: ['No' 'Yes' 'No internet service']

OnlineBackup: ['Yes' 'No' 'No internet service']

DeviceProtection: ['No' 'Yes' 'No internet service']

TechSupport: ['No' 'Yes' 'No internet service']

StreamingTV: ['No' 'Yes' 'No internet service']

StreamingMovies: ['No' 'Yes' 'No internet service']

Contract: ['Month-to-month' 'One year' 'Two year']

PaperlessBilling: ['Yes' 'No']

PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

Churn: ['No' 'Yes']



In [4]:
# Map binary columns to 0 and 1
binary_columns = ['gender', 'Partner', 'Dependents', 
                  'PhoneService', 'PaperlessBilling', 'Churn']

binary_map = {'Yes': 1, 'No': 0, 'Female': 0, 'Male': 1}

for col in binary_columns:
    df[col] = df[col].map(binary_map)

print("Binary columns converted")
df[binary_columns].head()

Binary columns converted


,gender,Partner,Dependents,PhoneService,PaperlessBilling,Churn
0,0,1,0,0,1,0
1,1,0,0,1,0,0
2,1,0,0,1,1,1
3,1,0,0,0,0,0
4,0,0,0,1,1,1


In [5]:
# One-hot encode the multi-value columns
multi_columns = ['MultipleLines', 'InternetService', 'OnlineSecurity', 
                 'OnlineBackup', 'DeviceProtection', 'TechSupport',
                 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=multi_columns, drop_first=True)

print(f"Columns after encoding: {df.shape[1]}")
print(df.columns.tolist())

Columns after encoding: 31
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [6]:
# Feature 1 — Average monthly spend over their lifetime
# A customer paying £80 for 1 month vs £80 for 3 years are very different
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)

# Feature 2 — How many services does the customer use?
# More services = more embedded = less likely to leave
service_columns = ['PhoneService', 'MultipleLines_Yes', 'OnlineSecurity_Yes',
                   'OnlineBackup_Yes', 'DeviceProtection_Yes', 
                   'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes']

df['TotalServices'] = df[service_columns].sum(axis=1)

# Feature 3 — Is this a new customer? (first 12 months = high risk)
df['IsNewCustomer'] = (df['tenure'] <= 12).astype(int)

# Feature 4 — High value customer paying above median
df['HighValueCustomer'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)

print("New features created")
print(df[['AvgMonthlySpend', 'TotalServices', 'IsNewCustomer', 'HighValueCustomer']].head())

New features created
   AvgMonthlySpend TotalServices  IsNewCustomer  HighValueCustomer
0        14.925000             1              1                  0
1        53.985714             3              0                  0
2        36.050000             3              1                  0
3        40.016304             3              0                  0
4        50.550000             1              1                  1


In [7]:
df.to_csv('../data/churn_cleaned.csv', index=False)

print(f"Saved: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Ready for modelling")

Saved: 7043 rows, 35 columns
Ready for modelling
